# Phase 4 – Advanced RAG: Semantic Cache + Multi-hop Retrieval

This notebook covers both Phase 4 features:
- **Task 1** – Semantic Caching (speed up repeated queries)
- **Task 2** – Multi-hop Retrieval (handle complex multi-part questions)

**Prerequisites:** run `python scripts/run_phase1.py` first.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.rag_pipeline         import RAGPipeline
from src.ingestion.chunk_text import load_chunks

chunks   = load_chunks()
pipeline = RAGPipeline(retriever_mode='faiss').load(chunks=chunks)
print('Pipeline ready')

## Task 1 – Semantic Caching

How it works:
```
User question
     ↓
Embed → 768-dim vector
     ↓
Compare with all cached vectors (cosine similarity)
     ↓                              ↓
sim >= 0.92 → CACHE HIT          sim < 0.92 → CACHE MISS
return instantly (no API)         call Gemini → store result
```

In [ ]:
from src.retrieval.semantic_cache import SemanticCache, cached_rag_query

cache = SemanticCache(threshold=0.92)
cache.clear()
print('Cache ready (empty)')

### Step 1: First query – Cache Miss

In [ ]:
q1 = "What was IFC's net income for fiscal year 2024?"

t0 = time.perf_counter()
answer1, from_cache1 = cached_rag_query(q1, pipeline, cache)
t1 = time.perf_counter() - t0

print(f'Source: {"CACHE" if from_cache1 else "Gemini API"}')
print(f'Time:   {t1:.2f}s')
print(f'Answer: {answer1[:200]}')

### Step 2: Same topic, different phrasing – Cache Hit

In [ ]:
similar_queries = [
    "How much net income did IFC earn in 2024?",
    "Tell me IFC's profit for fiscal year 2024.",
    "What profit did IFC report in the 2024 annual report?",
]

timings = [('Original (API)', False, t1)]

for q in similar_queries:
    t0 = time.perf_counter()
    ans, from_cache = cached_rag_query(q, pipeline, cache)
    t  = time.perf_counter() - t0
    timings.append((q[:50], from_cache, t))
    status = 'CACHE HIT' if from_cache else 'API CALL'
    print(f'{status} | {t:.3f}s | {q[:55]}')

In [ ]:
# Visualise speed difference
labels  = [t[0][:40] for t in timings]
times   = [t[2] for t in timings]
sources = [t[1] for t in timings]
colors  = ['#4CAF50' if s else '#F44336' for s in sources]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(labels, times, color=colors, edgecolor='white')
for bar, t in zip(bars, times):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{t:.2f}s', va='center', fontsize=9)
p1 = mpatches.Patch(color='#4CAF50', label='Cache Hit')
p2 = mpatches.Patch(color='#F44336', label='Cache Miss')
ax.legend(handles=[p1, p2])
ax.set_xlabel('Time (seconds)')
ax.set_title('Semantic Cache: Response Time')
plt.tight_layout()
plt.show()

### Step 3: Explore cosine similarity scores

In [ ]:
from src.retrieval.retriever import embed_query

original = "What was IFC's net income for fiscal year 2024?"
orig_emb = embed_query(original)

test_queries = [
    "What was IFC's net income for fiscal year 2024?",   # identical
    "How much net income did IFC earn in 2024?",          # very similar
    "Tell me IFC's profit for fiscal year 2024.",         # similar
    "What is IFC's total revenue in 2024?",               # related
    "What is the capital adequacy ratio?",                # different topic
    "What is the weather today?",                         # unrelated
]

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

sims = [(q[:55], cosine(orig_emb, embed_query(q))) for q in test_queries]

qs   = [s[0] for s in sims]
vals = [s[1] for s in sims]
cols = ['#4CAF50' if v >= 0.92 else '#FF9800' if v >= 0.80 else '#F44336' for v in vals]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(qs, vals, color=cols, edgecolor='white')
ax.axvline(0.92, color='green', linestyle='--', label='Cache hit threshold (0.92)')
ax.set_xlim(0, 1.05)
ax.set_xlabel('Cosine Similarity')
ax.set_title('Semantic Similarity to Original Question')
ax.legend()
plt.tight_layout()
plt.show()

### Step 4: Cache statistics and CSV export

In [ ]:
cache.print_stats()

export_path = cache.export_csv()
df_cache = pd.read_csv(export_path)
df_cache[['query', 'answer_preview', 'hits', 'source_pages']]

---
## Task 2 – Multi-hop Retrieval

### Why multi-hop?

Simple question (single-hop is fine):
> *What was IFC net income in 2024?*

Complex question (needs multi-hop):
> *How did IFC net income change from 2023 to 2024, and what risk factors caused this?*

```
Complex question
       ↓
Gemini: decompose → [sub-Q1, sub-Q2, sub-Q3]
       ↓
Hop 1: retrieve(sub-Q1) → partial answer 1
       ↓  (partial answer 1 passed as context)
Hop 2: retrieve(sub-Q2) + context → partial answer 2
       ↓
Hop 3: retrieve(sub-Q3) + context → partial answer 3
       ↓
Gemini: synthesise all → FINAL ANSWER
```

In [ ]:
from src.retrieval.multihop_retriever import (
    MultiHopRetriever, needs_multihop, decompose_question
)

questions = [
    'What was IFC net income in 2024?',
    'How did IFC net income change between 2023 and 2024 and why?',
    'What are IFC total assets?',
    'Compare IFC equity and loan income and explain the difference.',
]

print(f'{"Question":<65} {"Multi-hop?"}')
print('-' * 80)
for q in questions:
    result = needs_multihop(q)
    icon   = 'YES' if result else 'no'
    print(f'{q[:63]:<65} {icon}')

In [ ]:
# Demonstrate question decomposition
complex_q = (
    "How did IFC's net income change between fiscal year 2023 and 2024, "
    "and what were the main risk factors that contributed to this change?"
)

print(f'Complex question: "{complex_q}"\n')
print('Gemini decomposes into sub-questions:')
sub_qs = decompose_question(complex_q, max_hops=3)
for i, sq in enumerate(sub_qs, 1):
    print(f'  Hop {i}: {sq}')

In [ ]:
# Run full multi-hop pipeline
retriever = MultiHopRetriever(pipeline, max_hops=3, top_k=4)
result    = retriever.run(complex_q)

print(f'Total hops: {result.total_hops}')
print('=' * 55)
for hop in result.hops:
    print(f'\nHop {hop.hop_number}: {hop.sub_question}')
    print(f'Pages: {[c["page_number"] for c in hop.retrieved_chunks]}')
    print(f'Partial: {hop.partial_answer[:200]}...')

In [ ]:
print('FINAL SYNTHESISED ANSWER:')
print('=' * 55)
print(result.final_answer)

In [ ]:
# Compare single-hop vs multi-hop
single = retriever.run_single_hop(complex_q)

print('SINGLE-HOP:')
print(single[:400])
print()
print('MULTI-HOP:')
print(result.final_answer[:400])

In [ ]:
# Visualise pages retrieved per hop
fig, axes = plt.subplots(1, result.total_hops, figsize=(14, 4), sharey=False)
if result.total_hops == 1:
    axes = [axes]

for ax, hop in zip(axes, result.hops):
    hop_pages = [c['page_number'] for c in hop.retrieved_chunks]
    scores    = [c.get('score', 0.5) for c in hop.retrieved_chunks]
    ax.barh(range(len(hop_pages)), scores, color='steelblue', edgecolor='white')
    ax.set_yticks(range(len(hop_pages)))
    ax.set_yticklabels([f'p.{p}' for p in hop_pages])
    ax.set_xlabel('Score')
    ax.set_title(f'Hop {hop.hop_number}\n"{hop.sub_question[:35]}..."', fontsize=8)
    ax.set_xlim(0, 1)

plt.suptitle('Pages Retrieved Per Hop', fontsize=11)
plt.tight_layout()
plt.show()